In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta import configure_spark_with_delta_pip
from delta.tables import DeltaTable
from pyspark.sql.window import Window
from pyspark.sql.functions import col, row_number, to_timestamp
from helpers import get_table, get_bronze
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from functools import reduce
print(pyspark.__version__)

3.5.8


In [2]:
builder = (
    SparkSession.builder
    .appName("user_bronze")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)

spark = configure_spark_with_delta_pip(
    builder,
    extra_packages=["org.postgresql:postgresql:42.7.3"]
).getOrCreate()

26/04/27 18:04:31 WARN Utils: Your hostname, CVPThuyTTD11-1 resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/04/27 18:04:31 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/thuyttd11/.local/lib/python3.8/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/thuyttd11/.ivy2/cache
The jars for the packages stored in: /home/thuyttd11/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.postgresql#postgresql added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-bf8ad13f-def5-4b98-886a-3f7fddccc429;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.3.2 in central
	found io.delta#delta-storage;3.3.2 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.postgresql#postgresql;42.7.3 in central
	found org.checkerframework#checker-qual;3.42.0 in central
:: resolution report :: resolve 393ms :: artifacts dl 16ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.3.2 from central in [default]
	io.delta#delta-storage;3.3.2 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	org.checkerframework#checker-qual;3.42.0 from central in [default]
	org.postgresql#postgresql;42.7.3 from central in [default]
	------------------------

In [3]:
# spark.stop()

# Subscription changes
Read Bronze subscription_changes data 

+ Remove duplicates based on the change_id + change_date as primary key (composite key) and Ingest time: the change_date should have have precision at seconds. 
+ Cast all columns to consistent Silver data types.
+ Normalize change_type to a standard controlled domain. 
+ Validate referential integrity for subscription_id, old_plan_id, and new_plan_id. 

## Read Bronze subscription_changes data

In [4]:
# Bronze layer
subscription_changes_bronze = "/home/thuyttd11/subscription-analytics/spark-data/bronze/subscription_changes"
df_bronze_subscription_changes = get_bronze(subscription_changes_bronze, spark=spark)
df_bronze_subscription_changes.show(5)

26/04/27 18:04:45 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
[Stage 8:>                                                          (0 + 1) / 1]

+---+---------+---------------+-----------+-----------+-----------+-----------+--------------------+--------------------+--------------------+
| id|change_id|subscription_id|old_plan_id|new_plan_id|change_type|change_date|         ingest_time|   source_identifier|            batch_id|
+---+---------+---------------+-----------+-----------+-----------+-----------+--------------------+--------------------+--------------------+
|  1|        1|         235624|          7|          8|    upgrade| 2024-11-16|2026-04-24 12:18:...|jdbc:postgresql:/...|app-2026042405173...|
|  2|        2|         160020|          6|          2|  downgrade| 2024-05-05|2026-04-24 12:18:...|jdbc:postgresql:/...|app-2026042405173...|
|  3|        3|         156853|         20|         18|    upgrade| 2024-09-26|2026-04-24 12:18:...|jdbc:postgresql:/...|app-2026042405173...|
|  4|        4|         248316|          9|          5|    upgrade| 2024-03-08|2026-04-24 12:18:...|jdbc:postgresql:/...|app-2026042405173...|

In [5]:
df_bronze_subscription_changes.count()

25000

## Remove duplicates based on the change_id + change_date

In [6]:
w = Window.partitionBy("change_id", "change_date").orderBy(F.col("ingest_time").desc())
df_ranked = df_bronze_subscription_changes.withColumn(
    "rn",
    F.row_number().over(w)
)

df1 = df_ranked.filter(F.col("rn") == 1).drop("rn")
df1_quarantine = df_ranked.filter(F.col("rn") > 1).drop("rn")

In [7]:
df1.columns

['id',
 'change_id',
 'subscription_id',
 'old_plan_id',
 'new_plan_id',
 'change_type',
 'change_date',
 'ingest_time',
 'source_identifier',
 'batch_id']

## Quarantine NULL

In [8]:
required_cols = [
    "change_id",
    "subscription_id",
    "old_plan_id",
    "new_plan_id",
    "change_type",
    "change_date",
    "ingest_time"
]

null_condition = None
for c in required_cols:
    cond = F.col(c).isNull()
    null_condition = cond if null_condition is None else (null_condition | cond)

# valid rows = no null in required columns
df2 = df1.filter(~null_condition)

# quarantine rows = at least one null in required columns
df2_quarantine = df1.filter(null_condition)


In [9]:
df2.count(), df2_quarantine.count()

(25000, 0)

## Cast all columns to consistent Silver data types

In [10]:
df3 = df2.select(
    F.col("change_id").cast("bigint").alias("change_id"),
    F.col("subscription_id").cast("bigint").alias("subscription_id"),
    F.col("old_plan_id").cast("int").alias("old_plan_id"),
    F.col("new_plan_id").cast("int").alias("new_plan_id"),
    F.col("change_type").cast("string").alias("change_type"),
    F.col("change_date").cast("timestamp").alias("change_date"),
    F.col("ingest_time").cast("timestamp").alias("ingest_time"),
)

## Ensure change_type values and old_plan_id and new_plan_id are in the plan_id

### Ensure change_type values

In [11]:
allowed_change_types = ["upgrade", "initial", "downgrade"]

df_checked_change_type = (
    df3
    .withColumn("change_type_clean", F.lower(F.trim(F.col("change_type"))))
)

df4 = (
    df_checked_change_type
    .filter(F.col("change_type_clean").isin(allowed_change_types))
    .drop("change_type")
    .withColumnRenamed("change_type_clean", "change_type")
)

df4_quarantine = (
    df_checked_change_type
    .filter(~F.col("change_type_clean").isin(allowed_change_types) | F.col("change_type_clean").isNull())
    .withColumn("quarantine_reason", F.lit("invalid change_type"))
    .drop("change_type")
    .withColumnRenamed("change_type_clean", "change_type")
)

In [12]:
# df4.count(), df4_quarantine.count()

In [13]:
df4.columns

['change_id',
 'subscription_id',
 'old_plan_id',
 'new_plan_id',
 'change_date',
 'ingest_time',
 'change_type']

### Ensure valid new_plan_id and old_plan_id

Must have subscription_plans silver table already

In [14]:
# Reference plan IDs
df_plan_ref = (
    spark.read
    .format("delta")
    .load("./silver/plans")
    .select("plan_id")
)

# Valid rows: both old_plan_id and new_plan_id must exist in df_plan_ref.plan_id
df5 = (
    df4
    .join(
        df_plan_ref.withColumnRenamed("plan_id", "old_plan_id"),
        on="old_plan_id",
        how="left_semi"
    )
    .join(
        df_plan_ref.withColumnRenamed("plan_id", "new_plan_id"),
        on="new_plan_id",
        how="left_semi"
    )
)

# Quarantine rows: rows from df4 that are NOT in df5
df5_quarantine = (
    df4
    .join(
        df5.select(*df4.columns),
        on=df4.columns,
        how="left_anti"
    )
)

# df5.count(), df5_quarantine.count()

In [15]:
df5.show(2)

+-----------+-----------+---------+---------------+-------------------+--------------------+-----------+
|new_plan_id|old_plan_id|change_id|subscription_id|        change_date|         ingest_time|change_type|
+-----------+-----------+---------+---------------+-------------------+--------------------+-----------+
|          8|          7|        1|         235624|2024-11-16 00:00:00|2026-04-24 12:18:...|    upgrade|
|          2|          6|        2|         160020|2024-05-05 00:00:00|2026-04-24 12:18:...|  downgrade|
+-----------+-----------+---------+---------------+-------------------+--------------------+-----------+
only showing top 2 rows



## Get all quarantine tables

In [16]:
quarantine_dfs = [
    df1_quarantine,
    df4_quarantine,
    df5_quarantine
]

df_quarantine_all = reduce(
    lambda df1, df2: df1.unionByName(df2, allowMissingColumns=True),
    quarantine_dfs
)
df_quarantine_all.count()

0

## Upsert strategy
Store output as Delta table silver_subscription_changes

In [17]:
subscription_changes_silver = df5

In [18]:
# Correct silver path
silver_path = "./silver/subscription_changes"

# Columns for silver table
silver_cols = subscription_changes_silver.columns
df_upsert = subscription_changes_silver.select(*silver_cols)

# Deduplicate incoming batch
# Keep latest record per (change_id, change_date) by ingest_time
w = (
    Window
    .partitionBy("change_id")
    .orderBy(F.col("change_date").desc(),F.col("ingest_time").desc())
)

df_upsert = (
    df_upsert
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

# Upsert into Delta
if DeltaTable.isDeltaTable(spark, silver_path):
    print("Delta table already exists")
    delta_target = DeltaTable.forPath(spark, silver_path)

    (
        delta_target.alias("target")
        .merge(
            df_upsert.alias("source"),
            """
            target.change_id = source.change_id
            AND target.change_date = source.change_date
            """
        )
        # Update only if incoming row is newer
        .whenMatchedUpdate(
            condition="source.ingest_time > target.ingest_time",
            set={c: f"source.{c}" for c in silver_cols}
        )
        .whenNotMatchedInsert(
            values={c: f"source.{c}" for c in silver_cols}
        )
        .execute()
    )

else:
    print("Delta table does not exist yet")
    (
        df_upsert
        .write
        .format("delta")
        .mode("overwrite")
        .save(silver_path)
    )


Delta table already exists


In [19]:
df_user_silver_read = (
    spark.read
    .format("delta")
    .load(silver_path)
)

df_user_silver_read.show(5)

+---------+---------------+-----------+-----------+-------------------+--------------------+-----------+
|change_id|subscription_id|old_plan_id|new_plan_id|        change_date|         ingest_time|change_type|
+---------+---------------+-----------+-----------+-------------------+--------------------+-----------+
|        1|         235624|          7|          8|2024-11-16 00:00:00|2026-04-24 12:18:...|    upgrade|
|        2|         160020|          6|          2|2024-05-05 00:00:00|2026-04-24 12:18:...|  downgrade|
|        3|         156853|         20|         18|2024-09-26 00:00:00|2026-04-24 12:18:...|    upgrade|
|        4|         248316|          9|          5|2024-03-08 00:00:00|2026-04-24 12:18:...|    upgrade|
|        5|         195984|         11|          7|2025-09-10 00:00:00|2026-04-24 12:18:...|    upgrade|
+---------+---------------+-----------+-----------+-------------------+--------------------+-----------+
only showing top 5 rows



In [20]:
spark.stop()